In [ ]:
import os
import ee
import geemap
import json
import requests
import folium
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import io
import zipfile

from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, HTML, clear_output
from PIL import Image
from pyproj import Transformer
from datetime import datetime, timezone


## Import the raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [ ]:
image_name = '2024-06_mimal_test_S2'

tif_base_dir = 'cookie-cutting/'
tif_path = os.path.join(tif_base_dir, image_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    imported_raster = raster.read(1)
    # Read the first three bands for an RGB preview
    rgb_raster = raster.read([1, 2, 3])
    rgb_raster = np.transpose(rgb_raster, (1, 2, 0))
    rgb_raster = rgb_raster.astype(np.float32)
    rgb_raster = (rgb_raster - rgb_raster.min()) / (rgb_raster.max() - rgb_raster.min() + 1e-9)
    # Get the metadata of the raster
    imported_raster_meta = raster.meta
    # Get the raster transform parameters
    raster_transform = raster.transform
    raster_bounds = raster.bounds

print("Shape of the raster (rows, columns):")
print(imported_raster.shape)
print("\n")

print("Raster metadata:")
print(imported_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(raster_transform)

# Plot the raster as RGB using the original coordinates
import matplotlib.pyplot as plt
plt.imshow(
    rgb_raster,
    extent=(raster_bounds.left, raster_bounds.right, raster_bounds.bottom, raster_bounds.top),
    origin='upper',
)
plt.title('Imported Raster')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [ ]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Base directory for the png image
png_base_dir = 'cookie-cutting/'

# Load the png image
image = Image.open(f'{png_base_dir}{image_name}.png')

# Convert the image to a numpy array
pixel_data = np.array(image)

# Plot the image data in the png to ensure they match
plt.imshow(pixel_data)
plt.title('PNG (with padding)')
plt.xlabel('Column Index')
plt.ylabel('Row Index')
plt.show()

## Get image information

In [ ]:
# Get image information
png_width, png_height = image.size
print(f'Image size: ({png_width}, {png_height})')

# Calculate the different in the width and height of the image and the raster
width_diff = png_width - imported_raster.shape[1]
height_diff = png_height - imported_raster.shape[0]
print(f'Difference in width: {width_diff}')
print(f'Difference in height: {height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
width_diff = width_diff - min_pad
height_diff = height_diff - min_pad
print(f'Difference in width after removing right padding: {width_diff}')
print(f'Difference in height after removing bottom padding: {height_diff}')

In [ ]:
json_path = os.path.join('cookie-cutting', f'{image_name}.json')
with open(json_path, 'r') as f:
    label_data = json.load(f)

# LabelMe stores box corners in PNG pixel coordinates, which include padding.
left_pad = (png_width - imported_raster.shape[1]) - min_pad
top_pad = (png_height - imported_raster.shape[0]) - min_pad

def png_pixel_to_lonlat(x_png, y_png):
    col = x_png - left_pad
    row = y_png - top_pad
    lon, lat = raster_transform * (col, row)
    return lon, lat

bbox_records = []
for shape in label_data['shapes']:
    (x1, y1), (x2, y2) = shape['points']
    x_min, x_max = sorted([x1, x2])
    y_min, y_max = sorted([y1, y2])

    corners = {
        'top_left': png_pixel_to_lonlat(x_min, y_min),
        'top_right': png_pixel_to_lonlat(x_max, y_min),
        'bottom_right': png_pixel_to_lonlat(x_max, y_max),
        'bottom_left': png_pixel_to_lonlat(x_min, y_max),
    }
    center_lon, center_lat = png_pixel_to_lonlat((x_min + x_max) / 2, (y_min + y_max) / 2)

    bbox_records.append({
        'label': shape['label'],
        'x_min_png': x_min,
        'y_min_png': y_min,
        'x_max_png': x_max,
        'y_max_png': y_max,
        'top_left_lon': corners['top_left'][0],
        'top_left_lat': corners['top_left'][1],
        'top_right_lon': corners['top_right'][0],
        'top_right_lat': corners['top_right'][1],
        'bottom_right_lon': corners['bottom_right'][0],
        'bottom_right_lat': corners['bottom_right'][1],
        'bottom_left_lon': corners['bottom_left'][0],
        'bottom_left_lat': corners['bottom_left'][1],
        'center_lon': center_lon,
        'center_lat': center_lat,
    })

bbox_df = pd.DataFrame(bbox_records)
bbox_df

In [ ]:
# Plot the centre points of the bounding boxes on the raster
plt.imshow(
    rgb_raster,
    extent=(raster_bounds.left, raster_bounds.right, raster_bounds.bottom, raster_bounds.top),
    origin='upper',
)
plt.scatter(bbox_df['center_lon'], bbox_df['center_lat'], c='red', s=5)
plt.title('Waterhole locations (bounding box centres)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()  

# Initialise GEE

You will need to authenticate your Google account the first time you run this. 

In [ ]:
# Initialize Earth Engine
try:
    ee.Initialize()
    print("Earth Engine already initialized")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine initialized")

## Create AOIs from bounding box centres

Use the centres of the labelled bounding boxes as the AOI seed points, then buffer each one into a rectangular region for downstream sampling.

In [ ]:
half_side_m = 750  # half the side length in metres

# Turn each bounding-box centre into a rectangular AOI
def bbox_center_to_rect(row):
    point = ee.Geometry.Point([float(row['center_lon']), float(row['center_lat'])])
    return ee.Feature(
        point.buffer(half_side_m).bounds(),
        {
            'lon': float(row['center_lon']),
            'lat': float(row['center_lat']),
            'label': row['label'],
        },
    )

rect_fc = ee.FeatureCollection([
    bbox_center_to_rect(row)
    for _, row in bbox_df.iterrows()
])
print(f'Created {bbox_df.shape[0]} AOIs from bounding-box centres')

### Plot the sampled AOIs

In [ ]:
basemap = geemap.Map(lite_mode=True)
basemap.add_basemap("SATELLITE")
basemap.set_center(134.6, -13.425, 11)  # lon, lat, zoom — roughly central North Australia

# Style: red outlines, no fill
basemap.addLayer(
    rect_fc.style(color='red', fillColor='FF000033', width=2),  # last 2 hex digits = alpha
    {},
    'Sampled AOIs',
)

# basemap

# Clear previous outputs before displaying new map
clear_output(wait=True)
basemap.to_html(filename='satellite_map3.html')
HTML('satellite_map3.html')

# Get Sentinel-2 Collection with Cloud Masking

This function creates a cloud mask for Sentinel-2 imagery.

There are other approaches to filter clouds, such as the approach listed here: [https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless](https://developers.google.com/earth-engine/tutorials/community/sentinel-2-s2cloudless).

In [ ]:
def maskS2clouds_CSPlus(image):
    """
    Mask Sentinel-2 using Cloud Score+.
    Assumes the CS+ bands ('cs', 'cs_cdf') have already been linked
    to the S2 collection via linkCollection().
    """
    # Use the cs_cdf band (cumulative distribution function variant)
    # is generally more robust than 'cs' for time-series work.
    # Threshold range: 0 (not clear) to 1 (clear).
    #   0.60 = permissive (more pixels kept, some haze/thin cloud)
    #   0.65 = balanced (Google's commonly recommended default)
    #   0.80+ = strict (clean composites, fewer observations)
    QA_BAND = 'cs_cdf'
    CLEAR_THRESHOLD = 0.65

    mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
    return (image.divide(10000)
                 .updateMask(mask)
                 .copyProperties(image, ['system:time_start']))

# Get Monthly Composites of Sentinel-2

## Define a function to get monthly composites

We want to create monthly composites of Sentinel-2 imagery. This function will filter the Sentinel-2 image collection by date and area of interest (AOI), apply the cloud mask, and then compute the median of each 'cloud-free' pixel for the month.

In [ ]:
# Bands written to every chip, in this exact order. Position is meaningful to
# everything downstream, so do not reorder without updating waterhole_seg/io_tiles.py.
OPTICAL_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
EXPORT_BANDS = OPTICAL_BANDS + ['n_obs']

# Written wherever a month has no clear observation at all. Chosen to be
# impossible for surface reflectance, so "no data" can never be mistaken for a
# dark pixel. Previously these came through as 0.0, which is indistinguishable
# from genuinely dark water and silently poisons any temporal statistic.
NODATA = -9999


def get_monthly_composites(start_date, end_date, aoi):
    """Build one Cloud Score+ masked monthly median per month with any imagery.

    Returns a list of (date_str, ee.Image) pairs, date_str being 'YYYY-MM'. Each
    image carries OPTICAL_BANDS plus 'n_obs': the per-pixel count of clear
    observations that fed the median. A median over one surviving scene is a
    different measurement from a median over six, so the count is exported
    rather than guessed at later.

    Masked pixels are filled with NODATA (optical) and 0 (n_obs) so gaps are
    explicit in the file rather than implied by a zero.
    """
    s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(aoi)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 90)))

    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2_base.linkCollection(csPlus, ['cs', 'cs_cdf'])

    # ONE blocking round trip per site: the timestamps of every candidate scene,
    # from which the months that actually contain imagery are derived locally.
    # The previous version called .getInfo() twice inside the month loop, which
    # over a seven-year range is ~170 sequential round trips per site.
    stamps = s2_base.aggregate_array('system:time_start').getInfo()
    months_with_scenes = sorted({
        datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime('%Y-%m')
        for ms in stamps
    })
    print(f'  {len(stamps)} scenes spanning {len(months_with_scenes)} months')

    composites = []
    for date_str in months_with_scenes:
        month_start = ee.Date(f'{date_str}-01')
        monthly = (s2_linked
                     .filterDate(month_start, month_start.advance(1, 'month'))
                     .map(maskS2clouds_CSPlus))

        # Count first: it must be taken from the masked collection, before any
        # unmasking, or every pixel would report a full count.
        n_obs = monthly.select('B4').count().rename('n_obs').clip(aoi)
        median = monthly.median().select(OPTICAL_BANDS).clip(aoi)

        composite = (median.unmask(NODATA)
                           .addBands(n_obs.unmask(0))
                           .toFloat()          # one dtype -> clean multiband GeoTIFF
                           .select(EXPORT_BANDS))
        composites.append((date_str, composite))

    return composites


## Define a function to export and download images

If the images are small enough, you can download them directly to your local machine. If they are too large, you can export them to your Google Drive.

## For local export

In [ ]:
# Local download. Chips are ~150x150 px, so downloading them directly is far
# quicker end to end than round-tripping through Drive, provided the requests
# are issued concurrently rather than one at a time.

def _finalise_geotiff(path):
    """Stamp band names and the nodata value onto a downloaded chip.

    Earth Engine's GeoTIFF download carries neither. Without them band identity
    survives only as position, and every reader has to re-derive which value
    means "no observation". Both are written here, once, at the source.
    """
    with rasterio.open(path) as src:
        profile = src.profile
        data = src.read()

    if data.shape[0] != len(EXPORT_BANDS):
        raise ValueError(
            f'{path}: expected {len(EXPORT_BANDS)} bands, got {data.shape[0]}'
        )

    profile.update(nodata=NODATA)
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data)
        dst.descriptions = tuple(EXPORT_BANDS)


def download_composite(image, region, geotiff_path, scale=10, crs='EPSG:32753',
                       max_retries=3):
    """Download one composite to a GeoTIFF, with retries. Returns the path."""
    last_error = None
    for attempt in range(max_retries):
        try:
            url = image.getDownloadURL({
                'scale': scale,
                'crs': crs,
                'fileFormat': 'GeoTIFF',
                'filePerBand': False,
                # Explicit region: the image is unmasked, so image.geometry()
                # is no longer bounded by the AOI.
                'region': region,
            })
            response = requests.get(url, stream=True, timeout=300)
            response.raise_for_status()
            content = response.content

            if zipfile.is_zipfile(io.BytesIO(content)):
                with zipfile.ZipFile(io.BytesIO(content)) as archive:
                    tif_name = next(
                        (n for n in archive.namelist() if n.lower().endswith('.tif')),
                        None,
                    )
                    if tif_name is None:
                        raise ValueError('download archive contained no GeoTIFF')
                    with archive.open(tif_name) as source, open(geotiff_path, 'wb') as target:
                        target.write(source.read())
            else:
                with open(geotiff_path, 'wb') as f:
                    f.write(content)

            _finalise_geotiff(geotiff_path)
            return geotiff_path

        except Exception as error:  # noqa: BLE001 - retry on anything transient
            last_error = error
            if os.path.exists(geotiff_path):
                os.remove(geotiff_path)

    raise RuntimeError(f'failed after {max_retries} attempts: {last_error}')


def export_monthly_images(composites, region, folder_path, aoi_name, scale=10,
                          crs='EPSG:32753', max_workers=8, overwrite=False):
    """Download every monthly composite for one AOI as a multi-band GeoTIFF.

    Skips chips that already exist unless overwrite=True, so a run that dies
    partway can simply be restarted. Returns (n_written, n_skipped, failures).
    """
    os.makedirs(folder_path, exist_ok=True)

    pending = []
    n_skipped = 0
    for date_str, image in composites:
        path = os.path.join(folder_path, f'{aoi_name}_{date_str}.tif')
        if not overwrite and os.path.exists(path) and os.path.getsize(path) > 0:
            n_skipped += 1
            continue
        pending.append((date_str, image, path))

    failures = []
    if pending:
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {
                pool.submit(download_composite, image, region, path, scale, crs): date_str
                for date_str, image, path in pending
            }
            for future in as_completed(futures):
                date_str = futures[future]
                try:
                    future.result()
                except Exception as error:  # noqa: BLE001
                    failures.append((date_str, str(error)))

    print(f'  wrote {len(pending) - len(failures)}, skipped {n_skipped} existing'
          + (f', FAILED {len(failures)}' if failures else ''))
    for date_str, message in failures:
        print(f'    FAILED {date_str}: {message}')

    return len(pending) - len(failures), n_skipped, failures


## For Google Drive export

In [ ]:
# Google Drive export. Only worth it if the local threaded download above is
# being rate limited; chips this small usually come down faster directly.
# Note: Drive-exported files still need _finalise_geotiff() run over them once
# downloaded, since the band names and nodata tag are written client-side.

def export_to_drive(composites, region, aoi_name, drive_folder='sentinel2_images',
                    scale=10, crs='EPSG:32753'):
    """Start one Drive export task per monthly composite. Returns the tasks."""
    tasks = []
    for date_str, image in composites:
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=f'{aoi_name}_{date_str}',
            folder=drive_folder,
            fileNamePrefix=f'{aoi_name}_{date_str}',
            scale=scale,
            region=region,
            crs=crs,
            fileFormat='GeoTIFF',
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)

    print(f'  started {len(tasks)} Drive tasks for {aoi_name}')
    return tasks


def finalise_directory(folder_path):
    """Run _finalise_geotiff over every chip in a folder (for Drive downloads)."""
    from pathlib import Path
    paths = sorted(Path(folder_path).glob('*.tif'))
    for path in paths:
        _finalise_geotiff(str(path))
    print(f'finalised {len(paths)} chips in {folder_path}')


# Download images for a specified date range

We will get an image for June 2024 as an example.

In [ ]:
# --- export parameters ---------------------------------------------------

# Seven full years. The single 2024 year exported previously is not enough to
# support the temporal self-normalisation the classifier depends on: a harmonic
# fit has five free parameters, and the wet-season peak that sets its amplitude
# falls in the months worst affected by cloud. It also makes a multi-year
# degradation trend estimable at all, which one year does not.
START_DATE = '2019-01-01'
END_DATE = '2026-01-01'

# UTM zone 53S. Every site sits between 134.4E and 134.7E, which is inside zone
# 53 (132E-138E), so one zone covers the lot. The previous export used
# EPSG:4326, where a 10 m pixel is 10.0 m north-south but only ~9.7 m east-west
# at this latitude, so pixels were not square on the ground and pixel counts
# were not proportional to area.
#
# Labels are painted against the raster grid, so this must be settled BEFORE
# any labelling starts. Changing it later invalidates every label mask.
EXPORT_CRS = 'EPSG:32753'
EXPORT_SCALE = 10

# New directory: the new chips are on a different grid to the old ones and the
# two must not be mixed. cookie-cutting/images_tif is left untouched.
OUTPUT_FOLDER = 'cookie-cutting/images_tif_v2'


### Run the function

The function will print how many suitable images were found for the specified date range and AOI. If no images are found, you may need to adjust your date range or AOI.

In [ ]:
# Grab the first AOI from the bounding-box-derived feature collection
first_aoi_feature = ee.Feature(rect_fc.first())
single_aoi = first_aoi_feature.geometry()

single_name = 'first_aoi_test'
print(f'Testing first AOI: {single_name}')

# get_monthly_composites now returns a list of (date_str, ee.Image) pairs
composites = get_monthly_composites(START_DATE, END_DATE, single_aoi)
print(f'{len(composites)} monthly composites')
print('first / last:', composites[0][0], '/', composites[-1][0])


## Plot one of the randomly sampled AOIs

In [ ]:
# Visualise the first AOI and a handful of its monthly composites
basemap = geemap.Map()
basemap.centerObject(single_aoi, zoom=15)
basemap.add_basemap('SATELLITE')
basemap.addLayer(single_aoi, {}, 'First AOI')

viz_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25}

# Show a sparse sample rather than all ~84 months, which would be unusable
sample = composites[:: max(1, len(composites) // 12)]
for index, (date_str, image) in enumerate(sample):
    basemap.addLayer(image, viz_params, f'S2 RGB {date_str}', shown=(index == 0))
print(f'Added {len(sample)} of {len(composites)} composites to the map.')

clear_output(wait=True)
basemap.to_html(filename='satellite_map2.html')
HTML('satellite_map2.html')


# Export images to Google Drive

## Set up parameters for exporting the images

In [ ]:
# Convert the FeatureCollection to a list and get the count
rect_list = rect_fc.toList(rect_fc.size())
n = rect_fc.size().getInfo()
print(f'Number of AOIs: {n}')

lons = rect_fc.aggregate_array('lon').getInfo()
lats = rect_fc.aggregate_array('lat').getInfo()

def coord_tag(lat, lon):
    """e.g. lat=-42.8826, lon=147.3257 -> 'S42p88_E147p33'"""
    ns = 'S' if lat < 0 else 'N'
    ew = 'W' if lon < 0 else 'E'
    return f"{ns}{abs(lat):.2f}_{ew}{abs(lon):.2f}".replace('.', 'p')

base_name = image_name  # preserve the original AOI name for clarity

# Export images

## Local export

In [ ]:
# Full export. Resumable: chips that already exist on disk are skipped, so if
# this dies partway (or you interrupt it) just run the cell again.
#
# Scale: 187 sites x ~84 months is ~15,700 chips at roughly 100 KB each, so
# expect a few GB and a run measured in hours. Each site's months are fetched
# concurrently; sites are processed in order so progress is easy to read.

print(f'Exporting {n} AOIs, {START_DATE} to {END_DATE}, into {OUTPUT_FOLDER}')

all_exports = []
all_failures = []

for i in range(n):
    single_aoi = ee.Feature(rect_list.get(i)).geometry()
    tag = coord_tag(lats[i], lons[i])
    single_name = f'{base_name}_{i:03d}_{tag}'

    print(f'\n[{i + 1}/{n}] {single_name}')

    try:
        composites = get_monthly_composites(START_DATE, END_DATE, single_aoi)
        n_written, n_skipped, failures = export_monthly_images(
            composites,
            region=single_aoi,
            folder_path=OUTPUT_FOLDER,
            aoi_name=single_name,
            scale=EXPORT_SCALE,
            crs=EXPORT_CRS,
        )
        all_exports.append(single_name)
        all_failures.extend((single_name, date_str, message)
                            for date_str, message in failures)
    except Exception as error:  # noqa: BLE001
        print(f'  SITE FAILED: {error}')
        all_failures.append((single_name, 'ALL', str(error)))
        continue

print(f'\nCompleted {len(all_exports)} of {n} AOIs.')
if all_failures:
    print(f'{len(all_failures)} failures - re-run this cell to retry them:')
    for site, date_str, message in all_failures[:20]:
        print(f'  {site} {date_str}: {message}')


## Google Drive export

In [ ]:
# print(f'Processing {n} AOIs for Drive export...')

# all_tasks = []

# for i in range(n):
#     single_aoi = ee.Feature(rect_list.get(i)).geometry()
#     tag = coord_tag(lats[i], lons[i])
#     single_name = f'{base_name}_{i:03d}_{tag}'   
    
#     print(f'\n[{i+1}/{n}] AOI: {single_name}')

#     try:
#         composites = get_monthly_composites(start_date, end_date, single_aoi)
#         tasks = export_to_drive(
#             composites,
#             aoi=single_aoi,
#             aoi_name=single_name,
#             drive_folder='sentinel2_images',
#             scale=10,
#         )
#         all_tasks.extend(tasks)
#     except Exception as e:
#         print(f'  FAILED for {single_name}: {e}')
#         continue

# print(f'\nStarted {len(all_tasks)} total Drive export tasks.')

### To check the status of the downloads

In [ ]:
# print(f'Saved AOI exports: {len(all_exports)}') if 'all_exports' in globals() else print(f'Drive export tasks: {len(all_tasks)}')

# AlphaEarth annual embeddings

Google's Satellite Embedding summarises a year of multi-sensor observation into 64
dimensions per 10 m pixel. Added here rather than in a separate notebook so the embeddings
are cut from the same AOIs, with the same site ids, as the Sentinel-2 chips.

**These are grid-locked to the S2 chips.** Rather than asking for `scale=10` and hoping the
two grids coincide, each site's export is given the `crs`, `crs_transform` and `dimensions`
read from its existing Sentinel-2 chip. That makes the embedding raster pixel-identical to
the reflectance raster by construction, so the two can be stacked without resampling —
which matters, because resampling a learned embedding is not meaningful in the way
resampling reflectance is.

The embeddings are annual and static, so a single year is enough: **2025**.

In [ ]:
EMBEDDING_COLLECTION = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
EMBEDDING_BANDS = [f'A{i:02d}' for i in range(64)]

EMBEDDING_YEAR = 2025
EMBEDDING_FOLDER = 'cookie-cutting/AlphaEarth_tif'

# Where to read the grid from. Every embedding chip is written onto the exact
# grid of that site's Sentinel-2 chips.
S2_TILE_DIR = OUTPUT_FOLDER

print(f'{len(EMBEDDING_BANDS)} bands: {EMBEDDING_BANDS[0]}..{EMBEDDING_BANDS[-1]}')
print(f'year {EMBEDDING_YEAR} -> {EMBEDDING_FOLDER}')

## Read the grid to lock onto

One Sentinel-2 chip per site is enough — every month of a site shares one grid, which the
inventory verifies.

In [ ]:
import glob


def s2_grid_for_site(aoi_name, tile_dir=None):
    """The exact grid of a site's Sentinel-2 chips, as Earth Engine export params.

    Returns a dict with crs, crs_transform and dimensions. Passing these instead
    of `scale` pins the output to the same pixels as the reflectance data, so the
    embedding never needs resampling to line up.
    """
    tile_dir = tile_dir or S2_TILE_DIR
    matches = sorted(glob.glob(os.path.join(tile_dir, f'{aoi_name}_*.tif')))
    if not matches:
        raise FileNotFoundError(
            f'no Sentinel-2 chips for {aoi_name} in {tile_dir}; export those first'
        )

    with rasterio.open(matches[0]) as dataset:
        transform = dataset.transform
        return {
            'crs': str(dataset.crs),
            'crs_transform': [transform.a, transform.b, transform.c,
                              transform.d, transform.e, transform.f],
            'dimensions': f'{dataset.width}x{dataset.height}',
            'shape': (dataset.height, dataset.width),
        }


# Sanity check on the first AOI
_probe = s2_grid_for_site(f'{base_name}_000_{coord_tag(lats[0], lons[0])}')
print('grid for site 000:')
for key, value in _probe.items():
    print(f'  {key}: {value}')

## Build the embedding image for one AOI

In [ ]:
def get_annual_embedding(year, aoi):
    """One AlphaEarth image for a year, mosaicked and unmasked.

    No cloud masking: each annual image is already a gap-free summary of the year.
    mosaic() covers AOIs that straddle two source tiles.
    """
    yearly = (ee.ImageCollection(EMBEDDING_COLLECTION)
                .filterBounds(aoi)
                .filterDate(f'{year}-01-01', f'{year + 1}-01-01'))

    if yearly.size().getInfo() == 0:
        raise ValueError(f'no AlphaEarth image for {year} over this AOI')

    return (yearly.mosaic()
                  .select(EMBEDDING_BANDS)
                  .unmask(NODATA)          # same convention as the S2 chips
                  .toFloat())

## Download, grid-locked

Same shape as the Sentinel-2 downloader: threaded, resumable, and the band names and nodata
tag written locally because Earth Engine's GeoTIFF download carries neither.

In [ ]:
def _finalise_embedding(path, expected_shape):
    """Stamp band names and nodata, and refuse a chip that is off-grid.

    The shape assertion is the point of the whole grid-locking exercise: if the
    embedding does not land on exactly the S2 pixels, that must be an error here
    rather than a silent misalignment discovered much later.
    """
    with rasterio.open(path) as src:
        profile = src.profile
        data = src.read()
        shape = (src.height, src.width)

    if data.shape[0] != len(EMBEDDING_BANDS):
        raise ValueError(
            f'{path}: expected {len(EMBEDDING_BANDS)} bands, got {data.shape[0]}'
        )
    if shape != expected_shape:
        raise ValueError(
            f'{path}: shape {shape} does not match the Sentinel-2 grid '
            f'{expected_shape}. The crs_transform/dimensions export params did '
            f'not take effect.'
        )

    profile.update(nodata=NODATA)
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(data)
        dst.descriptions = tuple(EMBEDDING_BANDS)


def download_embedding(image, grid, geotiff_path, max_retries=3):
    """Download one embedding chip onto a fixed grid, with retries."""
    last_error = None
    for _ in range(max_retries):
        try:
            url = image.getDownloadURL({
                'fileFormat': 'GeoTIFF',
                'filePerBand': False,
                # crs_transform + dimensions fully determine the output grid, so
                # no 'scale' and no 'region' — either would fight with them.
                'crs': grid['crs'],
                'crs_transform': grid['crs_transform'],
                'dimensions': grid['dimensions'],
            })
            response = requests.get(url, stream=True, timeout=600)
            response.raise_for_status()
            content = response.content

            if zipfile.is_zipfile(io.BytesIO(content)):
                with zipfile.ZipFile(io.BytesIO(content)) as archive:
                    tif_name = next(
                        (n for n in archive.namelist() if n.lower().endswith('.tif')), None
                    )
                    if tif_name is None:
                        raise ValueError('download archive contained no GeoTIFF')
                    with archive.open(tif_name) as source, open(geotiff_path, 'wb') as target:
                        target.write(source.read())
            else:
                with open(geotiff_path, 'wb') as f:
                    f.write(content)

            _finalise_embedding(geotiff_path, grid['shape'])
            return geotiff_path

        except Exception as error:  # noqa: BLE001 - retry on anything transient
            last_error = error
            if os.path.exists(geotiff_path):
                os.remove(geotiff_path)

    raise RuntimeError(f'failed after {max_retries} attempts: {last_error}')

## Run the full export

187 AOIs, one year, 64 bands each. Resumable: chips already on disk are skipped, so
re-running the cell retries only what failed.

In [ ]:
os.makedirs(EMBEDDING_FOLDER, exist_ok=True)

print(f'Exporting AlphaEarth {EMBEDDING_YEAR} for {n} AOIs into {EMBEDDING_FOLDER}')

embedding_failures = []
n_written = n_skipped = 0
start_time = datetime.now()

for i in range(n):
    single_aoi = ee.Feature(rect_list.get(i)).geometry()
    tag = coord_tag(lats[i], lons[i])
    single_name = f'{base_name}_{i:03d}_{tag}'
    output_path = os.path.join(EMBEDDING_FOLDER, f'{single_name}_{EMBEDDING_YEAR}.tif')

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        n_skipped += 1
        continue

    try:
        grid = s2_grid_for_site(single_name)
        image = get_annual_embedding(EMBEDDING_YEAR, single_aoi)
        download_embedding(image, grid, output_path)
        n_written += 1
    except Exception as error:  # noqa: BLE001
        embedding_failures.append((single_name, str(error)))

    if (i + 1) % 20 == 0:
        elapsed = (datetime.now() - start_time).total_seconds()
        print(f'  {i + 1}/{n}  written {n_written}, skipped {n_skipped}, '
              f'failed {len(embedding_failures)}  ({elapsed:.0f}s)')

print(f'\nwrote {n_written}, skipped {n_skipped} existing, '
      f'{len(embedding_failures)} failed')
for name, message in embedding_failures[:10]:
    print(f'  FAILED {name}: {message[:110]}')

## Verify the grids are identical

The whole point of `crs_transform` was to make this check pass. If any chip disagrees with
its Sentinel-2 counterpart on CRS, transform or shape, the two cannot be stacked and the
embedding features would be silently offset.

In [ ]:
import numpy as np

mismatches = []
checked = 0
nodata_fractions = []

for i in range(n):
    tag = coord_tag(lats[i], lons[i])
    single_name = f'{base_name}_{i:03d}_{tag}'
    embedding_path = os.path.join(EMBEDDING_FOLDER, f'{single_name}_{EMBEDDING_YEAR}.tif')
    if not os.path.exists(embedding_path):
        mismatches.append((single_name, 'embedding chip missing'))
        continue

    s2_matches = sorted(glob.glob(os.path.join(S2_TILE_DIR, f'{single_name}_*.tif')))
    if not s2_matches:
        mismatches.append((single_name, 'no Sentinel-2 chip to compare against'))
        continue

    with rasterio.open(s2_matches[0]) as s2, rasterio.open(embedding_path) as ae:
        checked += 1
        if s2.crs != ae.crs:
            mismatches.append((single_name, f'crs {ae.crs} != {s2.crs}'))
        elif s2.transform != ae.transform:
            mismatches.append((single_name, 'transform differs'))
        elif s2.shape != ae.shape:
            mismatches.append((single_name, f'shape {ae.shape} != {s2.shape}'))
        elif ae.count != len(EMBEDDING_BANDS):
            mismatches.append((single_name, f'{ae.count} bands, expected 64'))
        else:
            band = ae.read(1)
            nodata_fractions.append(float(np.mean(~np.isfinite(band) | (band == NODATA))))

print(f'checked {checked} embedding chips against their Sentinel-2 grids')
if mismatches:
    print(f'!! {len(mismatches)} mismatch(es):')
    for name, issue in mismatches[:10]:
        print(f'   {name}: {issue}')
else:
    print('all grids identical: crs, transform, shape and band count')

if nodata_fractions:
    print(f'nodata per chip: median {100 * np.median(nodata_fractions):.2f}%, '
          f'max {100 * max(nodata_fractions):.2f}%')